# Local integrations
Use fse where the text already is. These integrations need no model download, credentials or paid API.
The tiny hand-built table demonstrates integration correctness only.
Install the pinned `requirements.txt`. Every section below runs against the named library, without fallback mocks.
Database adapters belong in application code, outside fse.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())
os.environ["DSPY_CACHEDIR"] = str(work / "dspy-cache")
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

In [ ]:
# Deliberately tiny, hand-built word rows. These demonstrate APIs, not model quality.
words = ["cat", "dog", "pet", "kitten", "puppy", "car", "bus", "road", "train", "truck"]
vectors = np.array([
    [1, 0, .2], [1, .1, 0], [1, 0, 0], [1, 0, .3], [1, .2, 0],
    [0, 1, .2], [0, 1, .1], [0, 1, 0], [0, 1, .3], [0, 1, .4],
], dtype=np.float32)
encoder = fse.Encoder.from_arrays(vectors, words=words, normalize=False)
texts = ["cat pet", "dog puppy", "car road", "bus truck"]

## DSPy
Supply a local callable to [DSPy's embedding retriever](https://github.com/stanfordnlp/dspy/blob/main/dspy/retrievers/embeddings.py).
This uses retrieval only. It configures no language model.

In [ ]:
import dspy

def embed(items):
    return encoder.encode(list(items), workers=1)
retriever = dspy.retrievers.Embeddings(corpus=texts, embedder=embed, k=1)
result = retriever("cat pet")
assert result.passages == [texts[0]]
print("DSPy:", result.passages)

## BERTopic
Pass [precomputed embeddings](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html).
A small deterministic PCA and KMeans example avoids downloading a default embedding model.
Use UMAP/HDBSCAN on a suitable corpus when density-based topics are wanted.

In [ ]:
from bertopic import BERTopic
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

documents = [f"{a} {b}" for group in (words[:5], words[5:]) for a in group for b in group]
topic_model = BERTopic(embedding_model=None, umap_model=PCA(n_components=2),
                     hdbscan_model=KMeans(n_clusters=2, random_state=7, n_init=10),
                     vectorizer_model=CountVectorizer(), calculate_probabilities=False)
topics, _ = topic_model.fit_transform(documents, embeddings=embed(documents))
assert len(topics) == len(documents) and len(set(topics)) == 2
print("BERTopic:", sorted(set(topics)))

## txtai
Configure a local vector callable. See [txtai vector configuration](https://neuml.github.io/txtai/embeddings/configuration/vectors/).
It receives a batch of texts and returns the batch matrix.

In [ ]:
from txtai import Embeddings
index = Embeddings({"method": "external", "transform": embed, "content": True})
index.index([(i, text, None) for i, text in enumerate(texts)])
hits = index.search("cat pet", 1)
assert hits[0]["id"] == "0" or hits[0]["id"] == 0
print("txtai:", hits)
index.close()

## Chroma
Give [explicit embeddings](https://docs.trychroma.com/docs/embeddings/embedding-functions) on insertion and query.
Disable the default embedding function to prevent an implicit model download.

In [ ]:
import chromadb
from chromadb.config import Settings
chroma = chromadb.EphemeralClient(Settings(anonymized_telemetry=False))
collection = chroma.create_collection("fse-recipe", embedding_function=None)
collection.add(ids=[str(i) for i in range(len(texts))], documents=texts, embeddings=embed(texts).tolist())
hits = collection.query(query_embeddings=embed(["cat pet"]).tolist(), n_results=1)
assert hits["ids"] == [["0"]]
print("Chroma:", hits["ids"])

## LanceDB
Store explicit rows in a temporary [local LanceDB table](https://docs.lancedb.com/search/vector-search).
No embedding registry or remote connection is required.

In [ ]:
import lancedb
lance = lancedb.connect(str(work / "lance"))
table = lance.create_table("texts", [{"id": i, "text": text, "vector": row.tolist()} for i, (text, row) in enumerate(zip(texts, embed(texts)))])
hits = table.search(embed(["cat pet"])[0]).limit(1).to_list()
assert hits[0]["id"] == 0
print("LanceDB:", hits[0]["text"])

## Qdrant and the public sink protocol
Implement `fse.Sink.target` and `commit` in application code.
The [local Qdrant client](https://github.com/qdrant/qdrant-client) runs in memory.
This adapter intentionally promises no `durable_rows`: memory-only upserts are not crash-durable.
Stable IDs make a repeated run idempotent. Each run uses a fresh sink instance.

In [ ]:
from qdrant_client import QdrantClient, models
client = QdrantClient(":memory:")
client.create_collection("texts", vectors_config=models.VectorParams(size=encoder.output_dim, distance=models.Distance.COSINE))

class QdrantSink:
    def __init__(self, client, collection, dim):
        self.client, self.collection, self.dim = client, collection, dim
        self.next_row = 0
    def target(self, first_row, n_rows):
        return np.empty((n_rows, self.dim), dtype=np.float32)
    def commit(self, first_row, rows):
        if first_row != self.next_row:
            raise ValueError("commits must be contiguous")
        self.client.upsert(self.collection, points=[models.PointStruct(id=first_row+i, vector=row.tolist()) for i, row in enumerate(rows)], wait=True)
        self.next_row += len(rows)

sink: fse.Sink = QdrantSink(client, "texts", encoder.output_dim)
for batch in encoder.encode_batches([texts[:2], texts[2:]], sink=sink, workers=1):
    assert batch.durable_rows is None
assert client.count("texts", exact=True).count == len(texts)
print("Qdrant sink points:", client.count("texts", exact=True).count)
client.close()

## Polars
Pass the Series directly through Arrow's public interface. Add the matrix as a fixed-size array column.
For lazy work use [batch functions](https://docs.pola.rs/user-guide/expressions/user-defined-python-functions/) rather than a Python call per row.

In [ ]:
import polars as pl
frame = pl.DataFrame({"text": texts})
rows = encoder.encode(frame["text"], workers=1)
np.testing.assert_array_equal(rows, embed(texts))
frame = frame.with_columns(pl.Series("vector", rows, dtype=pl.Array(pl.Float32, encoder.output_dim)))
assert frame.height == len(texts)
print(frame)

In [ ]:
scratch.cleanup()